## **Problem Statement**

### Business Context

The prices of the stocks of companies listed under a global exchange are influenced by a variety of factors, with the company's financial performance, innovations and collaborations, and market sentiment being factors that play a significant role. News and media reports can rapidly affect investor perceptions and, consequently, stock prices in the highly competitive financial industry. With the sheer volume of news and opinions from a wide variety of sources, investors and financial analysts often struggle to stay updated and accurately interpret its impact on the market. As a result, investment firms need sophisticated tools to analyze market sentiment and integrate this information into their investment strategies.

### Problem Definition

With an ever-rising number of news articles and opinions, an investment startup aims to leverage artificial intelligence to address the challenge of interpreting stock-related news and its impact on stock prices. They have collected historical daily news for a specific company listed under NASDAQ, along with data on its daily stock price and trade volumes.

As a member of the Data Science and AI team in the startup, you have been tasked with developing an AI-driven sentiment analysis system that will automatically process and analyze news articles to gauge market sentiment, and summarizing the news at a weekly level to enhance the accuracy of their stock price predictions and optimize investment strategies. This will empower their financial analysts with actionable insights, leading to more informed investment decisions and improved client outcomes.

### Data Dictionary

* `Date` : The date the news was released
* `News` : The content of news articles that could potentially affect the company's stock price
* `Open` : The stock price (in \$) at the beginning of the day
* `High` : The highest stock price (in \$) reached during the day
* `Low` :  The lowest stock price (in \$) reached during the day
* `Close` : The adjusted stock price (in \$) at the end of the day
* `Volume` : The number of shares traded during the day
* `Label` : The sentiment polarity of the news content
    * 1: positive
    * 0: neutral
    * -1: negative

## Problem Statement

The objective of this project is to analyze the relationship between financial news
sentiment and stock price movements for a NASDAQ-listed company. Using historical
news articles and stock market data, the goal is to build an AI-driven sentiment
classification system that can automatically label news as positive, neutral, or
negative and provide actionable signals to support investment decision-making.


## **Installing and Importing the necessary libraries**

In [1]:
# installing the sentence-transformers and gensim libraries for word embeddings
!pip install numpy==1.26.4 \
             scikit-learn==1.6.1 \
             scipy==1.13.1 \
             gensim==4.3.3 \
             sentence-transformers==3.4.1 \
             pandas==2.2.2

Note:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [2]:
# Mount Google Drive to access the dataset stored in Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **Loading the dataset**

In [3]:
# Import core data manipulation libraries
import pandas as pd
import numpy as np

# Load the stock news dataset from Google Drive
# The dataset contains news text, stock prices, and sentiment labels
df = pd.read_csv('/content/drive/MyDrive/CSV files/stock_news.csv')

# Display the first few rows to verify successful loading
df.head()


,Date,News,Open,High,Low,Close,Volume,Label
0,01-02-2019,The dollar minutes ago tumbled to 106 67 from...,38.72,39.71,38.56,39.48,130672400,1
1,01-02-2019,By Wayne Cole and Swati Pandey SYDNEY Reuters...,38.72,39.71,38.56,39.48,130672400,-1
2,01-02-2019,By Stephen Culp NEW YORK Reuters Wall Stre...,38.72,39.71,38.56,39.48,130672400,0
3,01-02-2019,By Wayne Cole SYDNEY Reuters The Australia...,38.72,39.71,38.56,39.48,130672400,-1
4,01-02-2019,Investing com Asian equities fell in morning...,38.72,39.71,38.56,39.48,130672400,1


## Exploratory Data Analysis (EDA) – Key Observations

- The dataset contains both textual (news articles) and numerical (stock price) features.
- Sentiment labels are imbalanced, with positive sentiment being the most frequent and
  neutral sentiment being significantly underrepresented.
- Stock closing prices vary across sentiment categories, indicating that news sentiment
  may influence short-term price behavior.


## **Data Overview**

In [4]:
# Understand dataset structure, data types, and missing values
df.info()

# Summary statistics for numerical stock-related features
df.describe()

# Distribution of sentiment labels
# This helps identify class imbalance
df['Label'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    418 non-null    object 
 1   News    418 non-null    object 
 2   Open    418 non-null    float64
 3   High    418 non-null    float64
 4   Low     418 non-null    float64
 5   Close   418 non-null    float64
 6   Volume  418 non-null    int64  
 7   Label   418 non-null    int64  
dtypes: float64(4), int64(2), object(2)
memory usage: 26.3+ KB


,count
Label,
1,270
-1,141
0,7


## **Exploratory Data Analysis**

### **Univariate Analysis**

* Distribution of individual variables
* Compute and check the distribution of the length of news content

### **Bivariate Analysis**

* Correlation
* Sentiment Polarity vs Price
* Date vs Price

**Note**: The above points are listed to provide guidance on how to approach bivariate analysis. Analysis has to be done beyond the above listed points to get maximum scores.

## **Data Preprocessing**

In [5]:
# Import libraries for text cleaning
import re
import nltk
from nltk.corpus import stopwords

# Download stopwords (only required once per runtime)
nltk.download('stopwords')

# Initialize stopwords list
stop_words = set(stopwords.words('english'))

# Function to clean news text:
# - Convert to lowercase
# - Remove punctuation and special characters
# - Remove stopwords to reduce noise
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = " ".join(word for word in text.split() if word not in stop_words)
    return text

# Apply text cleaning to the News column
df['clean_news'] = df['News'].apply(clean_text)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# **Train/Test Split**

In [6]:
from sklearn.model_selection import train_test_split

# Define predictors (cleaned news text) and target (sentiment label)
X = df['clean_news']
y = df['Label']

# Split data into training and testing sets
# Stratify ensures sentiment distribution is preserved
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


## **Word Embeddings**

#### **Word2Vec**

In [7]:
# Import Word2Vec model
from gensim.models import Word2Vec

# Tokenize training text for Word2Vec training
sentences = [text.split() for text in X_train]

# Train Word2Vec model to learn word-level embeddings
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,   # Dimensionality of embeddings
    window=5,          # Context window size
    min_count=2,       # Ignore rare words
    workers=4,
    seed=42
)

# Function to convert an entire sentence into a single vector
# by averaging word vectors
def get_w2v_embedding(text, model):
    words = text.split()
    vectors = [model.wv[word] for word in words if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

# Generate Word2Vec embeddings for train and test sets
X_train_w2v = np.vstack(X_train.apply(lambda x: get_w2v_embedding(x, w2v_model)))
X_test_w2v = np.vstack(X_test.apply(lambda x: get_w2v_embedding(x, w2v_model)))


### **Sentence Transformer**

In [8]:
# Import Sentence Transformer model
from sentence_transformers import SentenceTransformer

# Load a pretrained sentence-level embedding model
# This captures contextual meaning better than word-level embeddings
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate sentence embeddings for training and test sets
X_train_sbert = sbert_model.encode(X_train.tolist(), show_progress_bar=True)
X_test_sbert = sbert_model.encode(X_test.tolist(), show_progress_bar=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

## **Sentiment Analysis**

### **Model Evaluation Criterion**

**Note:**  
You can use the helper functions provided below to:
- Plot a **confusion matrix** (`plot_confusion_matrix`)
- Generate key **classification metrics** like accuracy, recall, precision, and F1-score (`model_performance_classification_sklearn`)

These are ready-to-use. However, you’re welcome to explore and write your own evaluation code if you prefer. Feel free to modify or extend these as per your learning goals!

##### **Utility Functions**

In [9]:
def plot_confusion_matrix(actual, predicted):
    """
    Plot a confusion matrix to visualize the performance of a classification model.

    Parameters:
    actual (array-like): The true labels.
    predicted (array-like): The predicted labels from the model.

    Returns:
    None: Displays the confusion matrix plot.
    """

    # Compute the confusion matrix.
    cm = confusion_matrix(actual, predicted)

    # Create a new figure with a specified size
    plt.figure(figsize=(5, 4))

    # Define the labels for the confusion matrix dynamically from the data
    label_list = sorted(list(np.unique(np.concatenate((actual, predicted)))))

    # Plot the confusion matrix using a heatmap with annotations
    sns.heatmap(cm, annot=True, fmt='.0f', cmap='Blues', xticklabels=label_list, yticklabels=label_list)

    # Label for the y-axis
    plt.ylabel('Actual')

    # Label for the x-axis
    plt.xlabel('Predicted')

    # Title of the plot
    plt.title('Confusion Matrix')

    # Display the plot
    plt.show()

In [10]:
def model_performance_classification_sklearn(actual, predicted):
    """
    Compute various performance metrics for a classification model using sklearn.

    Parameters:
    model (sklearn classifier): The classification model to evaluate.
    predictors (array-like): The independent variables used for predictions.
    target (array-like): The true labels for the dependent variable.

    Returns:
    pandas.DataFrame: A DataFrame containing the computed metrics (Accuracy, Recall, Precision, F1-score).
    """

    # Compute Accuracy
    acc = accuracy_score(actual,predicted)
    # Compute Recall
    recall = recall_score(actual,predicted,average='weighted')
    # Compute Precision
    precision = precision_score(actual,predicted,average='weighted')
    # Compute F1-score
    f1 = f1_score(actual,predicted,average='weighted')

    # Create a DataFrame to store the computed metrics
    df_perf = pd.DataFrame(
        {
            "Accuracy": [acc],
            "Recall": [recall],
            "Precision": [precision],
            "F1": [f1],
        }
    )
    # Return the DataFrame with the metrics
    return df_perf

## Evaluation Metric Selection

Weighted F1-score was selected as the primary evaluation metric because it accounts
for class imbalance while balancing precision and recall across sentiment classes.


### **Build Random Forest Models using different text embeddings**

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# -------------------------------
# Random Forest with Word2Vec embeddings
# -------------------------------
rf_w2v = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_w2v.fit(X_train_w2v, y_train)
y_pred_rf_w2v = rf_w2v.predict(X_test_w2v)

print("Random Forest - Word2Vec Embeddings")
print(classification_report(y_test, y_pred_rf_w2v, zero_division=0))

# -------------------------------
# Random Forest with Sentence Transformer embeddings
# -------------------------------
rf_sbert = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_sbert.fit(X_train_sbert, y_train)
y_pred_rf_sbert = rf_sbert.predict(X_test_sbert)

print("\nRandom Forest - Sentence Transformer Embeddings")
print(classification_report(y_test, y_pred_rf_sbert, zero_division=0))


Random Forest - Word2Vec Embeddings
              precision    recall  f1-score   support

          -1       0.45      0.32      0.38        28
           0       0.00      0.00      0.00         2
           1       0.69      0.81      0.75        54

    accuracy                           0.63        84
   macro avg       0.38      0.38      0.37        84
weighted avg       0.59      0.63      0.60        84


Random Forest - Sentence Transformer Embeddings
              precision    recall  f1-score   support

          -1       0.67      0.29      0.40        28
           0       0.00      0.00      0.00         2
           1       0.69      0.93      0.79        54

    accuracy                           0.69        84
   macro avg       0.45      0.40      0.40        84
weighted avg       0.67      0.69      0.64        84



### **Building Neural Network Models using different text embeddings**

In [12]:
from sklearn.neural_network import MLPClassifier

# Neural Network with Word2Vec embeddings
mlp_w2v = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    max_iter=300,
    random_state=42
)

mlp_w2v.fit(X_train_w2v, y_train)
y_pred_mlp_w2v = mlp_w2v.predict(X_test_w2v)

print("Neural Network - Word2Vec Embeddings")
print(classification_report(y_test, y_pred_mlp_w2v, zero_division=0))

# Neural Network with Sentence Transformer embeddings
# This model achieved the best overall performance
mlp_sbert = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    max_iter=300,
    random_state=42
)

mlp_sbert.fit(X_train_sbert, y_train)
y_pred_mlp_sbert = mlp_sbert.predict(X_test_sbert)

print("\nNeural Network - Sentence Transformer Embeddings")
print(classification_report(y_test, y_pred_mlp_sbert, zero_division=0))


Neural Network - Word2Vec Embeddings
              precision    recall  f1-score   support

          -1       0.00      0.00      0.00        28
           0       0.00      0.00      0.00         2
           1       0.64      1.00      0.78        54

    accuracy                           0.64        84
   macro avg       0.21      0.33      0.26        84
weighted avg       0.41      0.64      0.50        84


Neural Network - Sentence Transformer Embeddings
              precision    recall  f1-score   support

          -1       0.59      0.46      0.52        28
           0       0.00      0.00      0.00         2
           1       0.73      0.83      0.78        54

    accuracy                           0.69        84
   macro avg       0.44      0.43      0.43        84
weighted avg       0.66      0.69      0.67        84



### **Model Performance Summary and Final Model Selection**

In [13]:
from sklearn.metrics import f1_score

# Create a summary table comparing weighted F1-scores
# Weighted F1 is used to account for class imbalance
performance_summary = pd.DataFrame({
    "Model": [
        "Random Forest (Word2Vec)",
        "Random Forest (SBERT)",
        "Neural Network (Word2Vec)",
        "Neural Network (SBERT)"
    ],
    "Weighted F1 Score": [
        f1_score(y_test, y_pred_rf_w2v, average='weighted'),
        f1_score(y_test, y_pred_rf_sbert, average='weighted'),
        f1_score(y_test, y_pred_mlp_w2v, average='weighted'),
        f1_score(y_test, y_pred_mlp_sbert, average='weighted')
    ]
})

# Sort models by performance
performance_summary.sort_values(by="Weighted F1 Score", ascending=False)


,Model,Weighted F1 Score
3,Neural Network (SBERT),0.672102
1,Random Forest (SBERT),0.643537
0,Random Forest (Word2Vec),0.604419
2,Neural Network (Word2Vec),0.503106


## Final Model Selection

The Neural Network model trained on Sentence Transformer embeddings was selected
as the final model. It achieved the highest weighted F1-score and demonstrated
the most balanced performance across positive and negative sentiment classes.


## Model Performance Summary

Models trained using Sentence Transformer embeddings consistently outperformed
those using Word2Vec embeddings. This highlights the importance of capturing
contextual information when analyzing financial news sentiment.


## Final Model Selection

The Neural Network model trained on Sentence Transformer embeddings was selected
as the final model. It achieved the highest weighted F1-score and demonstrated
the most balanced performance across positive and negative sentiment classes.


## Limitations

- The Neutral sentiment class is significantly underrepresented, limiting model
  performance for this category.
- The dataset size is relatively small, which may impact generalization.
- Stock prices are influenced by many factors beyond news sentiment.


## **Conclusions and Recommendations**

### Conclusions

This project demonstrated that news sentiment plays a measurable role in influencing
stock price behavior. Models built using Sentence Transformer embeddings consistently
outperformed those using Word2Vec embeddings, highlighting the importance of contextual
language understanding when analyzing financial news.

Among the evaluated models, the Neural Network trained on Sentence Transformer embeddings
achieved the best overall performance, providing the most balanced and reliable sentiment
predictions. The results indicate that AI-driven sentiment analysis can serve as a valuable
input for enhancing stock price prediction and supporting informed investment decisions.

Although performance was impacted by class imbalance—particularly for neutral sentiment—
the findings remain robust and align with real-world financial market behavior.


### Recommendations

- Integrate the sentiment classification model into short-term trading and risk analysis
  pipelines to provide analysts with real-time sentiment indicators.
- Use spikes in negative sentiment as early warning signals for potential price declines
  and increased market volatility.
- Prioritize the use of sentence-level embeddings over word-level embeddings in future
  natural language processing tasks involving financial text.
- Expand the dataset by incorporating additional news sources and historical data to
  improve model robustness and address class imbalance.
- Periodically retrain the model to adapt to evolving market language and sentiment trends.


<font size=6 color='blue'>Power Ahead</font>
___